In [25]:
!pip install groq -q
print("Libraries installed sucessfully")

Libraries installed sucessfully


In [26]:
import sqlite3
import pandas as pd
import os
from groq import Groq
import re
print("all libraries imported sucessfully")

all libraries imported sucessfully


In [27]:
import os
os.environ["GROQ_API_KEY"]="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"
client=Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL="llama-3.1-8b-instant"
print("Groq client initialized sucessfully")
print(f"using model:{MODEL}")

Groq client initialized sucessfully
using model:llama-3.1-8b-instant


In [28]:
import io
csv_data= """student_id,name,age,gender,subject,marks,attendance,grade
1,Aarav Sharma,20,Male,Mathematics,88,92,A
2,Priya Patel,21,Female,Science,76,85,B
3,Rohan Mehta,20,Male,Programming,95,98,A+
4,Sneha Iyer,22,Female,Mathematics,62,78,C
5,Arjun Nair,21,Male,Programming,91,94,A+
6,Divya Krishnan,20,Female,Science,83,88,A
7,Karan Singh,22,Male,Mathematics,74,81,B
8,Ananya Gupta,21,Female,Programming,89,96,A
9,Vikram Reddy,20,Male,Science,70,79,B
10,Pooja Sharma,22,Female,Mathematics,55,72,D
11,Aditya Kumar,21,Male,Programming,97,99,A+
12,Meera Nambiar,20,Female,Science,81,87,A
13,Rahul Desai,22,Male,Mathematics,68,80,C
14,Kavitha Rajan,21,Female,Programming,86,93,A
15,Nikhil Verma,20,Male,Science,77,84,B
16,Swathi Pillai,22,Female,Mathematics,90,95,A+
17,Manish Joshi,21,Male,Programming,73,82,B
18,Lavanya Menon,20,Female,Science,66,76,C
19,Suresh Babu,22,Male,Mathematics,82,89,A
20,Anjali Singh,21,Female,Programming,94,97,A+
21,Deepak Nair,20,Male,Science,79,86,B
22,Rekha Sharma,22,Female,Mathematics,58,73,D
23,Sanjay Patel,21,Male,Programming,88,91,A
24,Usha Iyer,20,Female,Science,84,90,A
25,Vijay Kumar,22,Male,Mathematics,71,83,B
26,Nandita Rao,21,Female,Programming,92,96,A+
27,Ashok Reddy,20,Male,Science,65,77,C
28,Sunita Gupta,22,Female,Mathematics,87,93,A
29,Ravi Krishnan,21,Male,Programming,78,88,B
30,Bhavna Mehta,20,Female,Science,93,98,A+"""

df=pd.read_csv(io.StringIO(csv_data))
print('dataset loaded:',len(df),"rows",len(df.columns),"columns")
print("\n forst 5 rows")
df.head()

dataset loaded: 30 rows 8 columns

 forst 5 rows


,student_id,name,age,gender,subject,marks,attendance,grade
0,1,Aarav Sharma,20,Male,Mathematics,88,92,A
1,2,Priya Patel,21,Female,Science,76,85,B
2,3,Rohan Mehta,20,Male,Programming,95,98,A+
3,4,Sneha Iyer,22,Female,Mathematics,62,78,C
4,5,Arjun Nair,21,Male,Programming,91,94,A+


In [29]:
conn=sqlite3.connect("college.db")
df.to_sql("students",conn,if_exists='replace',index=False)
print("database created:college.db")
print("Table 'students' created with 30 students records")
test_df=pd.read_sql_query("select count(*) as total_rows from students",conn)
print("verification: ",test_df['total_rows'][0],"rows")

database created:college.db
Table 'students' created with 30 students records
verification:  30 rows


In [30]:
def get_schema(conn, table_name="students"):
  """
  this function reads the structuring of a database table.
  it returns information about each column:name and data type"""

  cursor = conn.cursor()
  # Corrected PRAGMA statement: directly embed table_name as SQLite PRAGMA doesn't support '?' placeholder
  cursor.execute(f"PRAGMA table_info('{table_name}')")
  columns = cursor.fetchall()

  schema_lines = []
  schema_lines.append(f"Table: {table_name}")
  schema_lines.append('Columns:')

  for col in columns:
    # col[1] is column name, col[2] is data type
    schema_lines.append(f"- {col[1]} ({col[2]})")

  # Corrected SELECT statement with f-string for table name
  cursor.execute(f"SELECT * FROM {table_name} LIMIT 3")
  sample_rows = cursor.fetchall()

  if sample_rows:
    schema_lines.append("Sample rows (first 3):")
    for row in sample_rows:
      schema_lines.append(f"- {row}")
  else:
    schema_lines.append("No sample rows found.")

  return "\n".join(schema_lines)
schema=get_schema(conn)
print(schema)

Table: students
Columns:
- student_id (INTEGER)
- name (TEXT)
- age (INTEGER)
- gender (TEXT)
- subject (TEXT)
- marks (INTEGER)
- attendance (INTEGER)
- grade (TEXT)
Sample rows (first 3):
- (1, 'Aarav Sharma', 20, 'Male', 'Mathematics', 88, 92, 'A')
- (2, 'Priya Patel', 21, 'Female', 'Science', 76, 85, 'B')
- (3, 'Rohan Mehta', 20, 'Male', 'Programming', 95, 98, 'A+')


In [31]:
def generate_sql(user_question,schema_text,client,model):
  """
  sends the user's question and database schema to the Groq LLM.
  the LLM generates a sql quesry that answers """

  system_prompt=f""" you are an expert SQL assistant.
  you are connected to sqlite database with the following structure:
  {schema_text}
  rules you must follow:
  1.generate only a valid sqlite sql query
  2.do not include any explanation or text-only the sql query
  3.do not use markdown code blocks.return the raw sql query
  4.the table name is students
  5.only use column names that exis in the schema above
  6.use single quotes for string values in where cluauses
  7.if the user asks for top n,use order by marks desc limit n"""

  response=client.chat.completions.create(
      model=model,
      messages=[
          {"role":"system","content":system_prompt},
          {"role":"user","content":user_question}
      ],
      temperature=0.0
  )
  sql_query=response.choices[0].message.content
  return sql_query
question="show me all the female students"
print("qiestion:",question)
print(f"\ngenerating sql:")

sql=generate_sql(question,schema,client,MODEL)
print(f"\nsql query:\n{sql}")

qiestion: show me all the female students

generating sql:

sql query:
SELECT * FROM students WHERE gender = 'Female'


In [32]:
def execute_sql(sql_query,conn):
  """clean the ai-generated sql and execute it on the sqlite database
  return the results as pandas dataframe.

  parameters:"""

  clean_sql=sql_query.strip()
  clean_sql=re.sub(r'``sql\s*','',clean_sql)
  clean_sql=clean_sql.strip()
  try:
    result_df=pd.read_sql_query(clean_sql,conn)
    return result_df,None
  except Exception as e:
    return None,str(e)
print("Executing sql: ",sql)
result,error=execute_sql(sql,conn)
if error:
    print("Error: ",error)
else:
    print("query returned: ",len(result),"rows")
    print(result)


Executing sql:  SELECT * FROM students WHERE gender = 'Female'
query returned:  15 rows
    student_id            name  age  gender      subject  marks  attendance  \
0            2     Priya Patel   21  Female      Science     76          85   
1            4      Sneha Iyer   22  Female  Mathematics     62          78   
2            6  Divya Krishnan   20  Female      Science     83          88   
3            8    Ananya Gupta   21  Female  Programming     89          96   
4           10    Pooja Sharma   22  Female  Mathematics     55          72   
5           12   Meera Nambiar   20  Female      Science     81          87   
6           14   Kavitha Rajan   21  Female  Programming     86          93   
7           16   Swathi Pillai   22  Female  Mathematics     90          95   
8           18   Lavanya Menon   20  Female      Science     66          76   
9           20    Anjali Singh   21  Female  Programming     94          97   
10          22    Rekha Sharma   22  Female

In [33]:
def text_to_sql_agent(user_question,conn,client,model,verbose=True):
  """
  the main ai agent function,
  takes a user question in plain english and returns database results
  """
  print("="*60)
  print("user question: ",user_question)
  print("="*6)

  #step1:get the database schema
  if verbose:
    print("[step 1]:reading database schema .....")
  schema_text=get_schema(conn)
  if verbose:
    print("schema loaded sucessfully")
  if verbose:
    print("[step 2]:generating sql query.....")

  generated_sql=generate_sql(user_question,schema_text,client,model)
  if verbose:
    print("generated sql: ",generated_sql)
  if verbose:
    print("[step 3]:executing sql on the database...")
  result_df,error=execute_sql(generated_sql,conn)
  if error:
    print("sql execution error: ",error)
    return None,generated_sql
  if verbose:
    print("[step 4] query returned",len(result_df),"rows")
    print("\nresult: ")
    print("-"*80)
    print(result_df.to_string(index=False))
    print("-"*80)
    return result_df,generated_sql
result,sql_used=text_to_sql_agent(
    "show top 5 students in programming",
    conn,client,MODEL
)


user question:  show top 5 students in programming
[step 1]:reading database schema .....
schema loaded sucessfully
[step 2]:generating sql query.....
generated sql:  SELECT * FROM students WHERE subject = 'Programming' ORDER BY marks DESC LIMIT 5
[step 3]:executing sql on the database...
[step 4] query returned 5 rows

result: 
--------------------------------------------------------------------------------
 student_id         name  age gender     subject  marks  attendance grade
         11 Aditya Kumar   21   Male Programming     97          99    A+
          3  Rohan Mehta   20   Male Programming     95          98    A+
         20 Anjali Singh   21 Female Programming     94          97    A+
         26  Nandita Rao   21 Female Programming     92          96    A+
          5   Arjun Nair   21   Male Programming     91          94    A+
--------------------------------------------------------------------------------


In [34]:
result1, _=text_to_sql_agent(
    "show me all students who study mathematics",
    conn,client,MODEL
)

user question:  show me all students who study mathematics
[step 1]:reading database schema .....
schema loaded sucessfully
[step 2]:generating sql query.....
generated sql:  SELECT * FROM students WHERE subject = 'Mathematics'
[step 3]:executing sql on the database...
[step 4] query returned 10 rows

result: 
--------------------------------------------------------------------------------
 student_id          name  age gender     subject  marks  attendance grade
          1  Aarav Sharma   20   Male Mathematics     88          92     A
          4    Sneha Iyer   22 Female Mathematics     62          78     C
          7   Karan Singh   22   Male Mathematics     74          81     B
         10  Pooja Sharma   22 Female Mathematics     55          72     D
         13   Rahul Desai   22   Male Mathematics     68          80     C
         16 Swathi Pillai   22 Female Mathematics     90          95    A+
         19   Suresh Babu   22   Male Mathematics     82          89     A
       

In [35]:
result2, _ = text_to_sql_agent(
    "what is the average marks for each subject",
    conn,client,MODEL
)


user question:  what is the average marks for each subject
[step 1]:reading database schema .....
schema loaded sucessfully
[step 2]:generating sql query.....
generated sql:  SELECT subject, AVG(marks) FROM students GROUP BY subject
[step 3]:executing sql on the database...
[step 4] query returned 3 rows

result: 
--------------------------------------------------------------------------------
    subject  AVG(marks)
Mathematics        73.5
Programming        88.3
    Science        77.4
--------------------------------------------------------------------------------


In [36]:
result3,_=text_to_sql_agent(
    "show students who who scored more than 90 marks",
    conn,client,MODEL
)


user question:  show students who who scored more than 90 marks
[step 1]:reading database schema .....
schema loaded sucessfully
[step 2]:generating sql query.....
generated sql:  SELECT * FROM students WHERE marks > 90
[step 3]:executing sql on the database...
[step 4] query returned 6 rows

result: 
--------------------------------------------------------------------------------
 student_id         name  age gender     subject  marks  attendance grade
          3  Rohan Mehta   20   Male Programming     95          98    A+
          5   Arjun Nair   21   Male Programming     91          94    A+
         11 Aditya Kumar   21   Male Programming     97          99    A+
         20 Anjali Singh   21 Female Programming     94          97    A+
         26  Nandita Rao   21 Female Programming     92          96    A+
         30 Bhavna Mehta   20 Female     Science     93          98    A+
--------------------------------------------------------------------------------


In [37]:
result4,_=text_to_sql_agent(
    "how many male and female students are there?",
    conn,client,MODEL
)

user question:  how many male and female students are there?
[step 1]:reading database schema .....
schema loaded sucessfully
[step 2]:generating sql query.....
generated sql:  SELECT COUNT(CASE WHEN gender = 'Male' THEN 1 END) AS male_count, 
       COUNT(CASE WHEN gender = 'Female' THEN 1 END) AS female_count 
FROM students
[step 3]:executing sql on the database...
[step 4] query returned 1 rows

result: 
--------------------------------------------------------------------------------
 male_count  female_count
         15            15
--------------------------------------------------------------------------------


In [38]:
result5,_=text_to_sql_agent(
    "show female students who scored above 85 in science or programming order by marks",
    conn,client,MODEL
)

user question:  show female students who scored above 85 in science or programming order by marks
[step 1]:reading database schema .....
schema loaded sucessfully
[step 2]:generating sql query.....
generated sql:  SELECT * FROM students WHERE gender = 'Female' AND subject IN ('Science', 'Programming') AND marks > 85 ORDER BY marks DESC
[step 3]:executing sql on the database...
[step 4] query returned 5 rows

result: 
--------------------------------------------------------------------------------
 student_id          name  age gender     subject  marks  attendance grade
         20  Anjali Singh   21 Female Programming     94          97    A+
         30  Bhavna Mehta   20 Female     Science     93          98    A+
         26   Nandita Rao   21 Female Programming     92          96    A+
          8  Ananya Gupta   21 Female Programming     89          96     A
         14 Kavitha Rajan   21 Female Programming     86          93     A
------------------------------------------------

In [39]:
def generate_answer(user_question,query_results_df,client,model):
  """
  takes the original user question and the database results"""
  if query_results_df is None or len(query_results_df)==0:
    return "no results were found for your query"
  results_text=query_results_df.to_string(index=False)
  prompt=f"""the user asked:'{user_question}'
the database returned these results:{results_text}
please write a clear,friendly,2-3 sentence answer to the user's question based on these results.
be specific,mention actual names and numbers from the data.do not add information not present in the result"""

  response=client.chat.completions.create(
    model=model,
    messages=[{"role":"user","content":prompt}],
    temperature=0.3)
  return response.choices[0].message.content.strip()


In [45]:
def smart_text_to_sql_agent(user_question,conn,client,model):
  """
  enchanced agent that returns both a data table and a natural language answer
  """
  print("="*60)
  print("question: ",user_question)
  print("="*60)

  schema_text=get_schema(conn)
  print("generating sql...")
  generated_sql=generate_sql(user_question,schema_text,client,model)
  print("generated sql: ",generated_sql)

  result_df,error=execute_sql(generated_sql,conn)
  if error:
    print("sql execution error: ",error)
    return
  print("data: ",len(result_df),"rows returned")
  display(result_df)
  print("generating natural language answer...")
  answer=generate_answer(user_question,result_df,client,model)
  print("\nanswer: ")
  print(answer)
  print("="*60)

smart_text_to_sql_agent(
      "who are the top 5 students in programming",
      conn,client,MODEL
  )



question:  who are the top 5 students in programming
generating sql...
generated sql:  SELECT name FROM students WHERE subject = 'Programming' ORDER BY marks DESC LIMIT 5
data:  5 rows returned


,name
0,Aditya Kumar
1,Rohan Mehta
2,Anjali Singh
3,Nandita Rao
4,Arjun Nair


generating natural language answer...

answer: 
Based on our database, the top 5 students in programming are:

1. Aditya Kumar, 
2. Rohan Mehta, 
3. Anjali Singh, 
4. Nandita Rao, 
5. Arjun Nair.
